# Offline Evaluation Assignment — LangSmith

In this assignment you will run a complete **offline evaluation** workflow on LangSmith:

1. **Create a dataset** locally and export it as a CSV.
2. **Download the CSV** and upload it to LangSmith through the UI to create a dataset.
3. **Connect** back to that dataset from this notebook using its name and **extract** the examples.
4. **Define a target model** whose outputs you want to evaluate.
5. **Run two LLM-based evaluators** (Correctness and Conciseness) and **log an experiment** to LangSmith.

> **Offline evaluation** means you evaluate against a *fixed, curated dataset* with known reference answers — as opposed to *online* evaluation, which scores live production traces.

Work through the notebook **top to bottom**, one section at a time.


---
# Section 0 — Setup

Install dependencies, import libraries, and configure your Azure OpenAI and LangSmith credentials.


### 0.1 — Install dependencies

In [ ]:
!pip install -q langsmith openai pandas

### 0.2 — Imports

In [ ]:
import os
import json
import re

import pandas as pd
from openai import AzureOpenAI
from langsmith import Client

### 0.3 — Configure Azure OpenAI

Fill in your Azure OpenAI details below — **endpoint**, **API version**, **deployment name**, and **API key**.

> ⚠️ Replace the `YOUR_..._HERE` placeholders with your own values. Avoid sharing this notebook (or a Colab share link) while real keys are pasted in, since anyone with the link could use them.


In [ ]:
os.environ["OPENAI_API_TYPE"]    = "azure"
os.environ["OPENAI_API_BASE"]    = "https://YOUR-RESOURCE.openai.azure.com"   # <-- your endpoint
os.environ["OPENAI_API_VERSION"] = "2024-12-01-preview"
os.environ["OPENAI_API_KEY"]     = "YOUR_AZURE_OPENAI_API_KEY_HERE"           # <-- your key
os.environ["OPENAI_DEPLOYMENT"]  = "gpt-5"                                     # <-- your deployment name

azure = AzureOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_API_BASE"),
    api_version=os.getenv("OPENAI_API_VERSION"),
)
DEPLOYMENT = os.getenv("OPENAI_DEPLOYMENT")
print("Azure OpenAI client ready. Deployment:", DEPLOYMENT)

**Quick connection test** (optional but recommended):

> Note: `gpt-5` and o-series reasoning deployments require `max_completion_tokens` (not `max_tokens`) and only accept the default `temperature`. The calls in this notebook follow that convention so they work across model types.


In [ ]:
try:
    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_completion_tokens=2000,
    )
    print("Connection OK ->", resp.choices[0].message.content.strip())
except Exception as e:
    print("Connection failed:", e)

### 0.4 — Configure LangSmith

Paste your LangSmith API key below.


In [ ]:
os.environ["LANGSMITH_API_KEY"] = "YOUR_LANGSMITH_API_KEY_HERE"   # <-- your key

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
print("LangSmith client initialized.")

---
# Section 1 — Create the Evaluation Dataset (CSV)

We build a small question-answer dataset in memory, export it to a CSV file, and download it. Each row has:

- **`question`** — the input that will be sent to the model.
- **`reference_answer`** — the gold / expected answer used by the evaluators.


### 1.1 — Define the examples

Feel free to edit, add, or replace rows. Keep the two column names (`question`, `reference_answer`) so the upload mapping in Section 1.3 stays simple.


In [ ]:
examples = [
    {
        "question": "What is the capital of France?",
        "reference_answer": "Paris.",
    },
    {
        "question": "Who wrote the play 'Romeo and Juliet'?",
        "reference_answer": "William Shakespeare.",
    },
    {
        "question": "What is the boiling point of water at sea level in degrees Celsius?",
        "reference_answer": "100 degrees Celsius.",
    },
    {
        "question": "How many continents are there on Earth?",
        "reference_answer": "Seven.",
    },
    {
        "question": "What gas do plants primarily absorb during photosynthesis?",
        "reference_answer": "Carbon dioxide.",
    },
    {
        "question": "In which year did the first human land on the Moon?",
        "reference_answer": "1969.",
    },
    {
        "question": "What is the largest planet in our solar system?",
        "reference_answer": "Jupiter.",
    },
    {
        "question": "What is the chemical symbol for gold?",
        "reference_answer": "Au.",
    },
]

df = pd.DataFrame(examples, columns=["question", "reference_answer"])
print(f"Built dataset with {len(df)} examples.")
df

### 1.2 — Save the CSV and download it

In [ ]:
CSV_PATH = "offline_eval_dataset.csv"
df.to_csv(CSV_PATH, index=False)
print("Saved:", CSV_PATH)

# Trigger a browser download in Colab
from google.colab import files
files.download(CSV_PATH)

### 1.3 — Upload the CSV to LangSmith (UI steps)

Now switch to the **LangSmith web app** and create the dataset from your downloaded CSV:

1. Go to **https://smith.langchain.com** and open **Datasets & Experiments**.
2. Click **+ New Dataset** → choose **Upload CSV** (or **Import from CSV**).
3. Select the `offline_eval_dataset.csv` file you just downloaded.
4. Give the dataset a clear **name** — you'll need it in the next section (e.g. `offline-eval-qa`).
5. **Map the columns:**
   - `question` → **Input** key
   - `reference_answer` → **Output** key (reference)
6. Confirm and create the dataset.

> 📝 **Remember the exact dataset name** — Section 2 needs it verbatim.


---
# Section 2 — Connect to the Uploaded Dataset

Point this notebook at the dataset you just created in the UI, then pull the examples back down to confirm everything loaded correctly.


### 2.1 — Enter the dataset name

Set this to the **exact** name you gave the dataset in LangSmith.


In [ ]:
DATASET_NAME = "offline-eval-qa"   # <-- change to your dataset name

### 2.2 — Extract the examples

Fetch the examples from LangSmith and preview them. This both verifies the connection and shows the **input/output key names** the evaluators must use.


In [ ]:
examples = list(client.list_examples(dataset_name=DATASET_NAME))
print(f"Fetched {len(examples)} examples from dataset '{DATASET_NAME}'.\n")

for i, ex in enumerate(examples[:5], start=1):
    print(f"--- Example {i} ---")
    print("inputs :", dict(ex.inputs))
    print("outputs:", dict(ex.outputs))
    print()

if examples:
    print("Input keys :", list(examples[0].inputs.keys()))
    print("Output keys:", list(examples[0].outputs.keys()))

> If the **input/output key names** printed above are not `question` and `reference_answer` (for example, if the UI capitalised them), update the keys used in the target function and evaluators in the next sections to match.


---
# Section 3 — Define the Target Model

This is the system **being evaluated**. For each example, LangSmith passes the example's `inputs` dict to this function; it returns a dict of outputs that the evaluators will score.


In [ ]:
def target(inputs: dict) -> dict:
    """Answer the dataset question using Azure OpenAI."""
    # Robust to different input key names
    question = inputs.get("question") or inputs.get("Question") or next(iter(inputs.values()), "")

    response = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer accurately and concisely."},
            {"role": "user", "content": question},
        ],
        max_completion_tokens=2000,
    )
    return {"answer": response.choices[0].message.content.strip()}

# Smoke test on the first example
if examples:
    print(target(dict(examples[0].inputs)))

---
# Section 4 — Define the LLM-Based Evaluators

We define **two** LLM-as-judge evaluators. Each receives:

- `inputs` — the example inputs (e.g. `question`)
- `outputs` — what the **target model** returned (e.g. `answer`)
- `reference_outputs` — the gold answer from the dataset (e.g. `reference_answer`)

and returns a dict with a `key`, a numeric `score`, and a short `comment`.

A small helper parses the judge's JSON reply safely.


In [ ]:
def _parse_judge_json(text: str) -> dict:
    """Parse a JSON object from an LLM reply, tolerating extra prose / code fences."""
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return {}

### 4.1 — Evaluator 1: Correctness

Judges whether the model's answer is factually correct **with respect to the reference answer**. Returns `1` (correct) or `0` (incorrect).


In [ ]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question  = inputs.get("question", "")
    answer    = outputs.get("answer", "")
    reference = reference_outputs.get("reference_answer", "")

    prompt = f"""You are grading an answer against a reference answer.

Question:
{question}

Reference answer:
{reference}

Model answer:
{answer}

Is the model answer correct given the reference answer? Minor wording differences are fine as long as the meaning matches.

Return ONLY a JSON object:
{{"score": 1 or 0, "explanation": "<short reason>"}}"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
    )
    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "correctness",
        "score": int(parsed.get("score", 0)),
        "comment": parsed.get("explanation", ""),
    }

### 4.2 — Evaluator 2: Conciseness

Judges whether the answer is **concise and free of unnecessary filler**, independent of correctness. Returns `1` (concise) or `0` (verbose / padded).


In [ ]:
def conciseness(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    question = inputs.get("question", "")
    answer   = outputs.get("answer", "")

    prompt = f"""You are evaluating the conciseness of an answer.

Question:
{question}

Model answer:
{answer}

Is the answer concise and to the point, without filler, repetition, or unnecessary preamble?

Return ONLY a JSON object:
{{"score": 1 or 0, "explanation": "<short reason>"}}"""

    resp = azure.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=2000,
    )
    parsed = _parse_judge_json(resp.choices[0].message.content)

    return {
        "key": "conciseness",
        "score": int(parsed.get("score", 0)),
        "comment": parsed.get("explanation", ""),
    }

---
# Section 5 — Run the Evaluation and Log the Experiment

`client.evaluate` runs the **target model** over every example in the dataset, applies both evaluators, and logs the results to LangSmith as a new **experiment**.


In [ ]:
experiment_results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[correctness, conciseness],
    experiment_prefix="offline-llm-eval",
    max_concurrency=2,
)

print("Evaluation complete.")
print(experiment_results)

### 5.1 — View the results

Open **LangSmith → Datasets & Experiments → your dataset → Experiments** to inspect per-example scores for **correctness** and **conciseness**, compare runs, and drill into individual outputs.

You have now completed a full offline evaluation: built a dataset, uploaded it, connected to it, ran a target model, and scored it with two LLM-based metrics. ✅
